In [30]:
from dotenv import load_dotenv
load_dotenv()
import os
groq_api_key = os.getenv('groq_api_key')
groq_api_key[:9]
from langchain_groq import ChatGroq
llm = ChatGroq(
    api_key = groq_api_key,
    model = 'llama-3.1-8b-instant'
)

In [36]:
print(llm.invoke('Isaac Newton - I want his birth day , award list').content)

**Isaac Newton's Birth Details:**

- Date of birth: January 4, 1643
- Place of birth: Woolsthorpe, Lincolnshire, England

**Isaac Newton's Awards and Honors:**

1. **Knighted**: He was knighted by Queen Anne in 1705 for his services to science.
2. **Fellow of the Royal Society**: He was elected as a Fellow of the Royal Society in 1672, and later served as its President from 1703 until his death in 1727.
3. **President of the Royal Society**: He served as the President of the Royal Society from 1703 to 1727.
4. **Lucasian Professor of Mathematics**: He held the prestigious Lucasian Chair of Mathematics at Cambridge University from 1669 to 1701.
5. **Master of the Mint**: He served as the Master of the Royal Mint from 1696 until his death in 1727.

**Other notable honors and recognitions:**

1. **Newton Medal**: The Newton Medal, established in 1976, is an award given by the Institute of Physics for outstanding contributions to physics.
2. **Isaac Newton Telescope**: The Isaac Newton Tel

# Comma Separated Output format

In [37]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

In [38]:
output_perser = CommaSeparatedListOutputParser()
format_instruction = output_perser.get_format_instructions()
print(format_instruction)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [45]:
template = """ 
List down the top 10 concepts to learn from the following topic :
topic nam is - {topic_name}

output Instructions -
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
-don't give any extra data or explanation - just give as format_instructions
"""

In [46]:
from langchain_core.prompts import PromptTemplate

In [48]:
prompt = PromptTemplate.from_template(
    template = template,
    input_variable = ['topic_name']
)

In [49]:
print(prompt.format(topic_name = "GEN AI"))

 
List down the top 10 concepts to learn from the following topic :
topic nam is - GEN AI

output Instructions -
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
-don't give any extra data or explanation - just give as format_instructions



In [50]:
chain = prompt | llm

In [51]:
chain.invoke({'topic_name' : "GEN AI"}).content

'Training Data,Types of Generative AI,Neural Network Architectures,Deep Learning Algorithms,Generative Adversarial Networks (GANs),Recurrent Neural Networks (RNNs),Transformers,Attention Mechanisms,Predictive Modeling,Explainability and Transparency.'

# JSON OUTPUT 

In [52]:
template = """"
you are a helpful assistant, I will provide a scientist name, and you will give follow details about them in json format:
```json
{{
  "name": "Full name of the scientist",
  "date_of_birth": "Date of birth of the scientist",
  "bio": [
    "A brief biography point 1",
    "A brief biography point 2",
    "A brief biography point 3",
    "A brief biography point 4"
  ],
  "award": {{
    "name": "Name of the major award received",
    "year": "Year when the award was received",
    "reason": "Reason for receiving the award"
  }}
}}```

The scientist name is {scientist_name}

**instructions:**
 - **Output must be in valid json format**
 - Ensure the biography points are concise and informative
 - Do not include any additional information outside the specified fields

"""

In [53]:
prompt = PromptTemplate(
    template=template,
    input_variables=["scientist_name"]

)

In [54]:
llm_chain = prompt | llm

In [55]:
from langchain_core.output_parsers import JsonOutputParser

In [56]:
res = llm_chain.invoke({'scientist_name': 'Isaac Newton'})

In [58]:
print(res.content)

```json
{
  "name": "Sir Isaac Newton",
  "date_of_birth": "January 4, 1643",
  "bio": [
    "English mathematician and physicist who developed the laws of motion and universal gravitation.",
    "Made significant contributions to the field of optics, calculus, and mathematics.",
    "Wrote the influential book 'Philosophiæ Naturalis Principia Mathematica' which laid the foundation for classical mechanics.",
    "Served as a Member of Parliament and was a fellow of the Royal Society."
  ],
  "award": {
    "name": "Copley Medal",
    "year": "1705",
    "reason": "For his contributions to the field of mathematics"
  }
}
```


In [59]:

parser = JsonOutputParser()

parsed_res = parser.parse(res.content)

parsed_res

{'name': 'Sir Isaac Newton',
 'date_of_birth': 'January 4, 1643',
 'bio': ['English mathematician and physicist who developed the laws of motion and universal gravitation.',
  'Made significant contributions to the field of optics, calculus, and mathematics.',
  "Wrote the influential book 'Philosophiæ Naturalis Principia Mathematica' which laid the foundation for classical mechanics.",
  'Served as a Member of Parliament and was a fellow of the Royal Society.'],
 'award': {'name': 'Copley Medal',
  'year': '1705',
  'reason': 'For his contributions to the field of mathematics'}}

In [60]:
parsed_res['name']

'Sir Isaac Newton'

In [62]:
print(res.content)

```json
{
  "name": "Sir Isaac Newton",
  "date_of_birth": "January 4, 1643",
  "bio": [
    "English mathematician and physicist who developed the laws of motion and universal gravitation.",
    "Made significant contributions to the field of optics, calculus, and mathematics.",
    "Wrote the influential book 'Philosophiæ Naturalis Principia Mathematica' which laid the foundation for classical mechanics.",
    "Served as a Member of Parliament and was a fellow of the Royal Society."
  ],
  "award": {
    "name": "Copley Medal",
    "year": "1705",
    "reason": "For his contributions to the field of mathematics"
  }
}
```


In [63]:
raw_output = res.content

In [64]:
import json

In [65]:
if raw_output.startswith('```json'):
    raw_output = raw_output.removeprefix('```json').removesuffix('```').strip()
elif raw_output.startswith('```'):
    raw_output = raw_output.removeprefix('```').removesuffix('```')

output_json = json.loads(raw_output)


In [66]:
output_json['name']

'Sir Isaac Newton'

# Pydantatic Object

In [12]:
from pydantic import BaseModel, Field
from typing import List, Dict


In [13]:
class Scientist(BaseModel):
    name: str = Field("name of the scientist")
    date_of_birth: str = Field("date of birth of the scientist")
    bio: List[str] = Field("biography points of the scientist")
    award: Dict[str, str] = Field("award details of the scientist")

In [14]:
sc = Scientist(name='Isaac Newton', date_of_birth='1643', bio=['English physicist and mathematician', 'Considered one of the greatest scientists of all time'], award={'Nobel Prize': '1705'})

In [15]:
from langchain_core.output_parsers import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=Scientist)

In [16]:
format_instruction = parser.get_format_instructions()

In [17]:
print(format_instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"default": "name of the scientist", "title": "Name", "type": "string"}, "date_of_birth": {"default": "date of birth of the scientist", "title": "Date Of Birth", "type": "string"}, "bio": {"default": "biography points of the scientist", "items": {"type": "string"}, "title": "Bio", "type": "array"}, "award": {"additionalProperties": {"type": "string"}, "default": "award details of the scientist", "title": "Award", "type": "object"}}}
```


In [18]:
template = """
take the name of  scientist -  {name} and give the details t as per the instructions below:
{format_instruction}

- Do not include any additional information outside the specified fields
- Only provide the json response

 """

In [19]:
prompt = PromptTemplate(

    input_variables= ['name', 'format_instruction'],
    template= template
)

In [24]:
chain =  prompt | llm | parser

In [25]:
res = chain.invoke({'name': 'Albert Einstein', 'format_instruction': format_instruction})

In [28]:
print(res.name)

Albert Einstein
